In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
import os

USER = "root"
PASSWORD = "root"
HOST = "localhost"
PORT = "3306"
DATABASE = "EC"

In [2]:
files = {
    "olist_orders_dataset.csv": "orders",
    "olist_order_items_dataset.csv": "order_items",
    "olist_order_payments_dataset.csv": "order_payments",
    "olist_order_reviews_dataset.csv": "order_reviews",
    "olist_customers_dataset.csv": "customers",
    "olist_sellers_dataset.csv": "sellers",
    "olist_products_dataset.csv": "products",
    "olist_geolocation_dataset.csv": "geolocation",
    "product_category_name_translation.csv": "product_category_translation"
}

csv_folder = "E:\\1 SQL PROJECT\\data"

In [3]:
for filename, table_name in files.items():
    # Create a FRESH engine for every table
    engine = create_engine(
        f"mysql+mysqlconnector://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}",
        pool_pre_ping=True
    )
    
    filepath = os.path.join(csv_folder, filename)
    print(f"Loading {filename} → {table_name}...")
    
    df = pd.read_csv(filepath)
    
    try:
        # For geolocation, load in chunks to avoid timeout
        if table_name == "geolocation":
            df.to_sql(
                table_name,
                con=engine,
                if_exists="replace",
                index=False,
                chunksize=10000   # loads 10k rows at a time
            )
        else:
            df.to_sql(table_name, con=engine, if_exists="replace", index=False)
        
        print(f"Done. Rows loaded: {len(df)}")
    
    except Exception as e:
        print(f"FAILED on {table_name}: {e}")
    
    finally:
        engine.dispose()  # force close connection after each table

print("\nAll tables loaded successfully.")

Loading olist_orders_dataset.csv → orders...
Done. Rows loaded: 99441
Loading olist_order_items_dataset.csv → order_items...
Done. Rows loaded: 112650
Loading olist_order_payments_dataset.csv → order_payments...
Done. Rows loaded: 103886
Loading olist_order_reviews_dataset.csv → order_reviews...
Done. Rows loaded: 99224
Loading olist_customers_dataset.csv → customers...
Done. Rows loaded: 99441
Loading olist_sellers_dataset.csv → sellers...
Done. Rows loaded: 3095
Loading olist_products_dataset.csv → products...
Done. Rows loaded: 32951
Loading olist_geolocation_dataset.csv → geolocation...
Done. Rows loaded: 1000163
Loading product_category_name_translation.csv → product_category_translation...
Done. Rows loaded: 71

All tables loaded successfully.
